# AI Gateway Priority Routing — Test Suite

Tests for the JWT Citadel 3-priority PTU routing policy.

## Test Contracts (deployed via Bicep)

| Team | Priority | Model | TPM | PTU TPM | Monthly Quota |
|------|----------|-------|-----|---------|---------------|
| Alpha | P1 (prod) | gpt-4o | 500 | 300 | 5,000 |
| Beta | P2 (standard) | gpt-4o | 400 | 200 | 3,000 |
| Gamma | P3 (batch) | gpt-4o | 300 | 0 | 1,000 |

## Prerequisites

1. Deploy with `azd up` (creates APIM, Entra apps, blob storage)
2. Run `add_app_reg_secrets.sh` to create client secrets
3. Fill in the config cell below with `azd env` values

In [40]:
import requests
import json
import time
import msal
from dataclasses import dataclass, field
from typing import Optional
from IPython.display import display, HTML, Markdown
import subprocess

## Configuration

Load from `azd env` or fill manually.

In [74]:
def get_azd_env():
    """Read all azd env values into a dict."""
    try:
        result = subprocess.run(['azd', 'env', 'get-values'], capture_output=True, text=True, check=True)
        env = {}
        for line in result.stdout.strip().split('\n'):
            if '=' in line:
                key, val = line.split('=', 1)
                env[key.strip()] = val.strip().strip('"')
        return env
    except Exception as e:
        print(f"Could not read azd env: {e}")
        return {}

azd = get_azd_env()

# Gateway config
GATEWAY_URL = azd.get('APIM_GATEWAY_URL', 'https://YOUR-APIM.azure-api.net')
API_URL = azd.get('APIM_API_URL', 'https://YOUR-APIM.azure-api.net/inference/openai')
TENANT_ID = azd.get('TENANT_ID', 'YOUR-TENANT-ID')
AUDIENCE = 'https://cognitiveservices.azure.com'

# Team credentials (from azd env after add_app_reg_secrets.sh)
TEAMS = {
    'alpha': {
        'client_id': azd.get('TEAM_ALPHA_APP_ID', ''),
        'client_secret': azd.get('TEAM_ALPHA_SECRET', ''),
        'priority': 1,
        'tpm': 500,
        'ptu_tpm': 300,
        'monthly_quota': 5000,
    },
    'beta': {
        'client_id': azd.get('TEAM_BETA_APP_ID', ''),
        'client_secret': azd.get('TEAM_BETA_SECRET', ''),
        'priority': 2,
        'tpm': 400,
        'ptu_tpm': 200,
        'monthly_quota': 3000,
    },
    'gamma': {
        'client_id': azd.get('TEAM_GAMMA_APP_ID', ''),
        'client_secret': azd.get('TEAM_GAMMA_SECRET', ''),
        'priority': 3,
        'tpm': 300,
        'ptu_tpm': 0,
        'monthly_quota': 1000,
    },
}

MODEL = 'gpt-4.1-mini'
API_VERSION = '2025-03-01-preview'

print(f"Gateway: {GATEWAY_URL}")
print(f"API URL: {API_URL}")
print(f"Tenant:  {TENANT_ID}")
for name, t in TEAMS.items():
    print(f"  {name}: P{t['priority']} client_id={t['client_id'][:8]}... tpm={t['tpm']} ptu={t['ptu_tpm']}")

Gateway: https://apim-nkma3njmekt34.azure-api.net
API URL: https://apim-nkma3njmekt34.azure-api.net/inference/openai
Tenant:  c29d6c2b-f765-41b3-b2a2-971a14239dfd
  alpha: P1 client_id=bb90b0d2... tpm=500 ptu=300
  beta: P2 client_id=369d21be... tpm=400 ptu=200
  gamma: P3 client_id=0eada081... tpm=300 ptu=0


## Helper Functions

In [ ]:
_tokens = {}

def get_token(team_name: str) -> str:
    """Get a bearer token for a team using MSAL client credentials."""
    if team_name in _tokens:
        return _tokens[team_name]
    
    team = TEAMS[team_name]
    app = msal.ConfidentialClientApplication(
        client_id=team['client_id'],
        client_credential=team['client_secret'],
        authority=f'https://login.microsoftonline.com/{TENANT_ID}',
    )
    result = app.acquire_token_for_client(scopes=[f'{AUDIENCE}/.default'])
    if 'access_token' not in result:
        raise Exception(f"Token acquisition failed for {team_name}: {result.get('error_description', result)}")
    _tokens[team_name] = result['access_token']
    return _tokens[team_name]


@dataclass
class GatewayResponse:
    status: int
    team: str
    route_target: str = ''
    backend_type: str = ''
    caller_name: str = ''
    priority: str = ''
    tpm_limit: int = 0
    tpm_remaining: int = 0
    quota_limit: int = 0
    quota_remaining: int = 0
    ptu_utilization: str = ''
    ptu_consumed: int = 0
    ptu_limit: int = 0
    matched_identity: str = ''
    body: dict = field(default_factory=dict)
    error: str = ''


def send_request(team_name: str, model: str = MODEL, prompt: str = 'Say hello in one word.') -> GatewayResponse:
    """Send a chat completion request through the gateway."""
    token = get_token(team_name)
    url = f"{API_URL}/deployments/{model}/chat/completions?api-version={API_VERSION}"
    headers = {
        'Authorization': f'Bearer {token}',
        'Content-Type': 'application/json',
    }
    body = {
        'messages': [{'role': 'user', 'content': prompt}],
        'max_tokens': 10,
    }
    
    r = requests.post(url, headers=headers, json=body, timeout=30)
    h = r.headers
    
    def safe_int(val, default=0):
        try: return int(val)
        except: return default
    
    resp = GatewayResponse(
        status=r.status_code,
        team=team_name,
        route_target=h.get('x-route-target', ''),
        backend_type=h.get('x-backend-type', ''),
        caller_name=h.get('x-caller-name', ''),
        priority=h.get('x-caller-priority', ''),
        tpm_limit=safe_int(h.get('x-ratelimit-limit-tokens', 0)),
        tpm_remaining=safe_int(h.get('x-ratelimit-remaining-tokens', 0)),
        quota_limit=safe_int(h.get('x-quota-limit-tokens', 0)),
        quota_remaining=safe_int(h.get('x-quota-remaining-tokens', 0)),
        ptu_utilization=h.get('x-ptu-utilization', ''),
        ptu_consumed=safe_int(h.get('x-ptu-consumed', 0)),
        ptu_limit=safe_int(h.get('x-ptu-limit', 0)),
        matched_identity=h.get('x-caller-identity', ''),
    )
    
    try:
        resp.body = r.json()
    except:
        resp.body = {'raw': r.text[:200]}
    
    if r.status_code >= 400:
        err = resp.body.get('error', '')
        msg = resp.body.get('message', '')
        if isinstance(err, dict):
            resp.error = err.get('message', str(err))
        elif msg:
            resp.error = f"{err}: {msg}" if err else msg
        else:
            resp.error = str(err) if err else r.text[:200]
    
    return resp


def print_response(r: GatewayResponse):
    icon = '✅' if r.status == 200 else '⚠️' if r.status == 429 else '🚫' if r.status == 403 else '❌'
    print(f"{icon} [{r.team}] {r.status} | route={r.route_target} backend={r.backend_type} pri={r.priority} | "
          f"tpm: {r.tpm_remaining}/{r.tpm_limit} | quota: {r.quota_remaining}/{r.quota_limit} | "
          f"ptu: {r.ptu_consumed}/{r.ptu_limit} util={r.ptu_utilization}")
    if r.error:
        print(f"   ↳ {r.error}")


def send_n(team: str, n: int, delay: float = 0.5, **kwargs) -> list[GatewayResponse]:
    """Send N requests for a team, printing each result."""
    results = []
    for i in range(n):
        r = send_request(team, **kwargs)
        print_response(r)
        results.append(r)
        if i < n - 1:
            time.sleep(delay)
    return results

---
## Test 1: Config Viewer

Verify the `/gateway/config` endpoint renders the contract configuration.

In [77]:
config_url = f"{GATEWAY_URL}/gateway/config"
r = requests.get(config_url)
print(f"Status: {r.status_code}")
print(f"Content-Type: {r.headers.get('Content-Type')}")
if r.status_code == 200:
    display(HTML(r.text))
else:
    print(r.text[:500])

Status: 200
Content-Type: text/html; charset=utf-8


Team,Identities,Priority,Models (TPM / PTU),Monthly Quota,Env
Team Alpha,Peter K oid 93a15eb8-4106-4e48-b323-ae2cbbae7898Team Alpha App azp bb90b0d2-3d9c-4106-b711-4007b047ed32,P1 Production,gpt-4.1-mini 500 TPM 300 PTUgpt-oss-120b 100 TPM,"5,000",PROD
Team Beta,Team Beta (auto-created) azp 369d21be-7c13-450f-869d-55d14ec9406d,P2 Standard,gpt-4.1-mini 400 TPM 200 PTUgpt-5.1-chat 400 TPM,"3,000",DEV
Team Gamma,Team Gamma (auto-created) azp 0eada081-9004-47f5-b05f-96931c09a624,P3 Batch,gpt-4.1-mini 300 TPMgpt-4.1 100 TPM,"1,000",PROD


---
## Test 2: Identity & Contract Resolution

Each team should be recognized and get correct contract fields in response headers.

In [78]:
print("=== Identity Resolution ===")
for team in ['alpha', 'beta', 'gamma']:
    r = send_request(team)
    print_response(r)
    expected = TEAMS[team]
    assert r.status == 200, f"{team}: expected 200, got {r.status}: {r.error}"
    assert r.tpm_limit == expected['tpm'], f"{team}: tpm_limit {r.tpm_limit} != {expected['tpm']}"
    assert r.quota_limit == expected['monthly_quota'], f"{team}: quota_limit {r.quota_limit} != {expected['monthly_quota']}"
    assert r.ptu_limit == expected['ptu_tpm'], f"{team}: ptu_limit {r.ptu_limit} != {expected['ptu_tpm']}"
    print(f"   ✓ Contract verified: name={r.caller_name} identity={r.matched_identity}\n")

print("All identity tests passed! ✅")

=== Identity Resolution ===
✅ [alpha] 200 | route=pool pri=production | tpm: 484/500 | quota: 4984/5000 | ptu: 0/300 util=0.0%
   ✓ Contract verified: name=Team Alpha identity=Team Alpha App

✅ [beta] 200 | route=pool pri=standard | tpm: 384/400 | quota: 2984/3000 | ptu: 0/200 util=0.8%
   ✓ Contract verified: name=Team Beta identity=Team Beta (auto-created)

✅ [gamma] 200 | route=pool pri=batch | tpm: 284/300 | quota: 0/1000 | ptu: 0/0 util=1.6%
   ✓ Contract verified: name=Team Gamma identity=Team Gamma (auto-created)

All identity tests passed! ✅


---
## Test 3: Priority Routing

All priorities route to the unified `pool` (which contains both PTU and PAYG backends).
The pool's internal priority handles backend selection.

- **P1 (Alpha)**: priority=production, route=pool
- **P2 (Beta)**: priority=standard, route=pool
- **P3 (Gamma)**: priority=batch, route=pool

In [68]:
print("=== Priority Routing & Token Counters ===\n")

for team_name, label in [('alpha', 'P1/production'), ('beta', 'P2/standard'), ('gamma', 'P3/batch')]:
    expected = TEAMS[team_name]
    exp_priority = {1: 'production', 2: 'standard', 3: 'batch'}[expected['priority']]
    print(f"--- {team_name.upper()} ({label}) ---")

    results = []
    for i in range(3):
        r = send_request(team_name)
        print_response(r)
        results.append(r)
        assert r.status == 200, f"{team_name} req {i+1}: expected 200, got {r.status}: {r.error}"
        assert r.route_target == 'pool', f"{team_name}: expected route=pool, got {r.route_target}"
        assert r.priority == exp_priority, f"{team_name}: expected priority={exp_priority}, got {r.priority}"
        assert r.tpm_limit == expected['tpm'], f"{team_name}: tpm_limit {r.tpm_limit} != {expected['tpm']}"
        assert r.quota_limit == expected['monthly_quota'], f"{team_name}: quota_limit {r.quota_limit} != {expected['monthly_quota']}"
        assert r.ptu_limit == expected['ptu_tpm'], f"{team_name}: ptu_limit {r.ptu_limit} != {expected['ptu_tpm']}"
        time.sleep(0.3)

    # Verify overall counter movement (first vs last)
    first, last = results[0], results[-1]
    assert last.tpm_remaining < first.tpm_remaining, \
        f"{team_name}: tpm_remaining should decrease over 3 requests ({first.tpm_remaining} → {last.tpm_remaining})"
    assert last.quota_remaining < first.quota_remaining, \
        f"{team_name}: quota_remaining should decrease over 3 requests ({first.quota_remaining} → {last.quota_remaining})"
    if expected['ptu_tpm'] > 0:
        assert last.ptu_consumed > first.ptu_consumed, \
            f"{team_name}: ptu_consumed should increase over 3 requests ({first.ptu_consumed} → {last.ptu_consumed})"

    print(f"   ✓ {team_name}: route=pool pri={exp_priority} | tpm: {first.tpm_remaining}→{last.tpm_remaining} "
          f"quota: {first.quota_remaining}→{last.quota_remaining}\n")

print("All routing & counter tests passed! ✅")

=== Priority Routing & Token Counters ===

--- ALPHA (P1/production) ---
✅ [alpha] 200 | route=pool pri=production | tpm: 484/500 | quota: 4984/5000 | ptu: 0/300 util=0.0%
✅ [alpha] 200 | route=pool pri=production | tpm: 474/500 | quota: 4968/5000 | ptu: 16/300 util=0.8%
✅ [alpha] 200 | route=pool pri=production | tpm: 469/500 | quota: 4952/5000 | ptu: 32/300 util=1.6%
   ✓ alpha: route=pool pri=production | tpm: 484→469 quota: 4984→4952

--- BETA (P2/standard) ---
✅ [beta] 200 | route=pool pri=standard | tpm: 384/400 | quota: 2984/3000 | ptu: 0/200 util=2.4%
✅ [beta] 200 | route=pool pri=standard | tpm: 372/400 | quota: 2968/3000 | ptu: 16/200 util=3.2%
✅ [beta] 200 | route=pool pri=standard | tpm: 366/400 | quota: 2952/3000 | ptu: 32/200 util=4.0%
   ✓ beta: route=pool pri=standard | tpm: 384→366 quota: 2984→2952

--- GAMMA (P3/batch) ---
✅ [gamma] 200 | route=pool pri=batch | tpm: 284/300 | quota: 984/1000 | ptu: 0/0 util=4.8%
✅ [gamma] 200 | route=pool pri=batch | tpm: 271/300 | qu

---
## Test 4: Model Access Control

Teams should only access models in their contract. A request for an unlisted model should get 403.

In [69]:
print("=== Model Access Control ===")

# Alpha: gpt-4.1-mini allowed
r = send_request('alpha', model='gpt-4.1-mini')
print_response(r)
assert r.status == 200, f"Alpha/gpt-4.1-mini should be 200, got {r.status}"
print("   ✓ Alpha → gpt-4.1-mini allowed\n")

# Alpha: gpt-4o should be denied (not in test contract)
r = send_request('alpha', model='gpt-4o')
print_response(r)
assert r.status == 403, f"Alpha/gpt-4o should be 403, got {r.status}"
assert 'model_not_authorized' in str(r.body), f"Expected model_not_authorized error"
print("   ✓ Alpha → gpt-4o denied\n")

print("Model access tests passed! ✅")

=== Model Access Control ===
✅ [alpha] 200 | route=pool pri=production | tpm: 484/500 | quota: 4984/5000 | ptu: 48/300 util=4.8%
   ✓ Alpha → gpt-4.1-mini allowed

🚫 [alpha] 403 | route= pri= | tpm: 0/0 | quota: 0/0 | ptu: 0/0 util=
   ↳ model_not_authorized: Model 'gpt-4o' is not in your access contract. Allowed: gpt-4.1-mini, gpt-oss-120b
   ✓ Alpha → gpt-4o denied

Model access tests passed! ✅


---
## Test 5: Unknown Caller Rejection

A token from an unregistered app should get 401.

In [70]:
print("=== Unknown Caller ===")

url = f"{API_URL}/deployments/{MODEL}/chat/completions?api-version={API_VERSION}"
r = requests.post(url, headers={'Authorization': 'Bearer invalid-token', 'Content-Type': 'application/json'},
                  json={'messages': [{'role': 'user', 'content': 'test'}], 'max_tokens': 5}, timeout=10)
print(f"Status: {r.status_code}")
assert r.status_code == 401, f"Invalid token should get 401, got {r.status_code}"
print("   ✓ Invalid token → 401\n")

print("Unknown caller tests passed! ✅")

=== Unknown Caller ===
Status: 500


AssertionError: Invalid token should get 401, got 500

---
## Test 6: Per-Model TPM Rate Limiting

Gamma has only 300 TPM for gpt-4o. Rapid requests should trigger 429.

In [71]:
print("=== Per-Model TPM Rate Limit (Gamma: 300 TPM) ===")
print("Sending rapid requests to exhaust TPM...\n")

big_prompt = 'Write a detailed essay about the history of computing. ' * 5

results = send_n('gamma', 15, delay=0.2, prompt=big_prompt)

success = sum(1 for r in results if r.status == 200)
rate_limited = sum(1 for r in results if r.status == 429)

print(f"\nResults: {success} success, {rate_limited} rate-limited")
assert rate_limited > 0, "Expected at least one 429 with 300 TPM limit"
print("\nTPM rate limiting works! ✅")

=== Per-Model TPM Rate Limit (Gamma: 300 TPM) ===
Sending rapid requests to exhaust TPM...

✅ [gamma] 200 | route=pool pri=batch | tpm: 232/300 | quota: 932/1000 | ptu: 0/0 util=0.0%
✅ [gamma] 200 | route=pool pri=batch | tpm: 167/300 | quota: 864/1000 | ptu: 0/0 util=0.0%
✅ [gamma] 200 | route=pool pri=batch | tpm: 107/300 | quota: 796/1000 | ptu: 0/0 util=0.0%
✅ [gamma] 200 | route=pool pri=batch | tpm: 47/300 | quota: 728/1000 | ptu: 0/0 util=0.0%
✅ [gamma] 200 | route=pool pri=batch | tpm: 0/300 | quota: 660/1000 | ptu: 0/0 util=0.0%
⚠️ [gamma] 429 | route=unknown pri= | tpm: 0/0 | quota: 0/0 | ptu: 0/0 util=
   ↳ Token limit is exceeded. Try again in 2 seconds.
⚠️ [gamma] 429 | route=unknown pri= | tpm: 0/0 | quota: 0/0 | ptu: 0/0 util=
   ↳ Token limit is exceeded. Try again in 1 seconds.
✅ [gamma] 200 | route=pool pri=batch | tpm: 0/300 | quota: 592/1000 | ptu: 0/0 util=0.0%
⚠️ [gamma] 429 | route=unknown pri= | tpm: 0/0 | quota: 0/0 | ptu: 0/0 util=
   ↳ Token limit is exceeded

---
## Test 7: Monthly Quota Exhaustion

Gamma has only 1,000 monthly quota. Keep sending until exhausted.

In [79]:
print("=== Monthly Quota Exhaustion (Gamma: 1000 tokens) ===")
print("Sending requests until quota is exhausted...\n")

quota_hit = False
for i in range(50):
    r = send_request('gamma', prompt='Tell me a long story about dragons and knights. ' * 3)
    print_response(r)
    # llm-token-limit returns 403 (not 429) when monthly quota is exceeded
    if r.status == 403 and 'quota' in r.error.lower():
        print(f"\n   ✓ Monthly quota exhausted after {i+1} requests (403 — quota exceeded)")
        quota_hit = True
        break
    if r.status == 429 and r.quota_remaining <= 0:
        print(f"\n   ✓ Monthly quota exhausted after {i+1} requests (429 — rate limited)")
        quota_hit = True
        break
    time.sleep(1)  # wait for TPM window to reset between batches

if quota_hit:
    print("Monthly quota limiting works! ✅")
else:
    print("⚠️ Quota not exhausted in 50 requests — may need more/larger prompts")

=== Monthly Quota Exhaustion (Gamma: 1000 tokens) ===
Sending requests until quota is exhausted...

🚫 [gamma] 403 | route=unknown pri= | tpm: 300/0 | quota: 0/0 | ptu: 0/0 util=
   ↳ Token quota is exceeded. Try again in 14 days, 4 hours, 50 minutes, and 45 seconds.

   ✓ Monthly quota exhausted after 1 requests (403 — quota exceeded)
Monthly quota limiting works! ✅


---
## Test 8: PTU Soft Overflow

Alpha (P1) has 300 PTU TPM. When PTU consumption exceeds the PTU limit,
the pool's circuit breaker silently fails over to PAYG — requests keep
succeeding (200) instead of getting a hard 429.

We verify this by watching `x-ptu-consumed` grow past `x-ptu-limit`
while all requests continue to return 200.

In [ ]:
print("=== PTU Soft Overflow (Alpha: 300 PTU TPM) ===")
print("Sending rapid requests to exhaust PTU quota...\n")

results = send_n('alpha', 20, delay=0.3, prompt='Explain quantum computing in detail. ' * 3)

success = sum(1 for r in results if r.status == 200)
failed = sum(1 for r in results if r.status != 200)
ptu_count = sum(1 for r in results if r.backend_type == 'ptu')
payg_count = sum(1 for r in results if r.backend_type == 'payg')
peak_consumed = max((r.ptu_consumed for r in results), default=0)
ptu_limit = results[0].ptu_limit if results else 0

print(f"\nResults: {success} success, {failed} failed")
print(f"Backend routing: {ptu_count} PTU, {payg_count} PAYG")
print(f"PTU tracking: consumed up to {peak_consumed} / {ptu_limit} limit")

# All requests should succeed — pool circuit breaker handles PTU→PAYG silently
assert failed == 0, f"Expected all requests to succeed (soft overflow), but {failed} failed"
print("   ✓ All requests returned 200 (no hard 429 from PTU exhaustion)")

if payg_count > 0 and ptu_count > 0:
    print(f"   ✓ Observed PTU→PAYG overflow: {ptu_count} PTU then {payg_count} PAYG")
    print("\nPTU soft overflow works! ✅")
elif payg_count > 0:
    print(f"   PTU circuit breaker was already tripped — all {payg_count} went to PAYG")
elif ptu_count > 0:
    print(f"\n⚠️ All {ptu_count} requests served by PTU — PTU quota may not have been exhausted")
    print("   Try increasing prompt size or number of requests")

---
## Test 9: P2 Routing Under Load

When PTU utilization is high (>70%), P2 should be redirected to PAYG.
The PTU simulator is set to 30% — so P2 starts on PTU.
As P1 sends traffic, the cached utilization should climb and P2 should shift to PAYG.

In [ ]:
print("=== P2 Routing Under Load ===")
print("Step 1: Baseline — P2 should route to PTU at low utilization\n")

r = send_request('beta')
print_response(r)
baseline_route = r.route_target
print(f"   Baseline route: {baseline_route}")

print(f"\nStep 2: P1 (Alpha) flooding PTU to raise utilization...\n")
flood_results = send_n('alpha', 10, delay=0.2, prompt='Explain everything about machine learning. ' * 5)

print(f"\nStep 3: Check P2 routing after P1 flood...\n")
time.sleep(1)
r2 = send_request('beta')
print_response(r2)

print(f"\nBaseline: {baseline_route} → After flood: {r2.route_target}")
print(f"PTU utilization: {r2.ptu_utilization}")
print("\nNote: With the PTU simulator at 30% base, actual routing shift depends on")
print("how the simulated utilization interacts with real cache values.")

---
## Test 10: Cross-Team Isolation

Rate-limiting one team should NOT affect another team's quota.

In [ ]:
print("=== Cross-Team Isolation ===")
print("Step 1: Exhaust Gamma's TPM...\n")

big_prompt = 'Explain everything. ' * 10
gamma_results = send_n('gamma', 10, delay=0.1, prompt=big_prompt)
gamma_429 = sum(1 for r in gamma_results if r.status == 429)
print(f"\nGamma: {gamma_429} rate-limited\n")

print("Step 2: Alpha should still work fine...\n")
r = send_request('alpha')
print_response(r)
assert r.status == 200, f"Alpha should not be affected by Gamma's rate limit, got {r.status}"
print("\n   ✓ Alpha unaffected by Gamma's rate limiting")

print("\nStep 3: Beta should still work fine...\n")
r = send_request('beta')
print_response(r)
assert r.status == 200, f"Beta should not be affected by Gamma's rate limit, got {r.status}"
print("\n   ✓ Beta unaffected by Gamma's rate limiting")

print("\nCross-team isolation verified! ✅")

---
## Test 11: PTU Simulator — Capacity Exhaustion & 429

The smart PTU simulator tracks cumulative consumption across all teams.
With 2,000 TPM capacity and 30% base utilization, there's ~1,400 usable tokens.
Flooding with requests should trigger simulated 429s (header `x-ptu-simulated: true`).

In [ ]:
print("=== PTU Simulator — Capacity Exhaustion ===")
print("Simulator: 2,000 TPM, 30% base util → ~1,400 usable tokens")
print("Flooding P1 (Alpha) to exhaust simulated PTU...\n")

big_prompt = 'Write a comprehensive analysis of artificial intelligence. ' * 5

saw_simulated_200 = False
saw_simulated_429 = False

results = send_n('alpha', 25, delay=0.2, prompt=big_prompt)

for r in results:
    is_simulated = r.body.get('raw', '') if isinstance(r.body, dict) else ''
    if r.status == 200 and r.route_target == 'ptu-pool':
        saw_simulated_200 = True
    if r.status == 429 and 'PTU Simulator' in str(r.body):
        saw_simulated_429 = True

ptu_200 = sum(1 for r in results if r.status == 200 and r.route_target == 'ptu-pool')
ptu_429 = sum(1 for r in results if r.status == 429 and 'PTU Simulator' in str(r.body))
payg_200 = sum(1 for r in results if r.status == 200 and r.route_target == 'std-pool')
real_429 = sum(1 for r in results if r.status == 429 and 'PTU Simulator' not in str(r.body))

print(f"\nResults:")
print(f"  PTU 200s (simulated): {ptu_200}")
print(f"  PTU 429s (simulated): {ptu_429}")
print(f"  PAYG 200s (overflow): {payg_200}")
print(f"  Real 429s (TPM limit): {real_429}")

if saw_simulated_200:
    print("\n   ✓ Requests served from simulated PTU")
if saw_simulated_429:
    print("   ✓ Simulated PTU returned 429 when capacity exhausted")
    print("\nPTU simulator 429s work! ✅")
else:
    print("\n⚠️ No simulated 429s observed — capacity may not have been exhausted")
    print("   Try increasing prompt size or number of requests")

---
## Test 12: PTU Simulator — P2 Shift Under Rising Utilization

As P1 consumes simulated PTU capacity, the cached utilization rises.
P2 should start on PTU (util < 70%) then shift to PAYG (util ≥ 70%).

In [ ]:
print("=== P2 Shift Under Simulated PTU Load ===")
print("Wait 65s for simulator cache to reset (60s TTL)...")
time.sleep(65)

print("\nStep 1: Baseline — P2 should route to PTU (fresh cache, low util)")
r = send_request('beta')
print_response(r)
baseline = r.route_target

print(f"\nStep 2: P1 flooding to raise simulated utilization...")
big_prompt = 'Explain everything about deep learning architectures. ' * 8
flood = send_n('alpha', 15, delay=0.1, prompt=big_prompt)

print(f"\nStep 3: Check P2 routing after flood...")
time.sleep(1)

p2_routes = []
for i in range(5):
    r = send_request('beta')
    print_response(r)
    p2_routes.append(r.route_target)
    time.sleep(0.5)

ptu_routes = sum(1 for rt in p2_routes if rt == 'ptu-pool')
payg_routes = sum(1 for rt in p2_routes if rt == 'std-pool')

print(f"\nBaseline: {baseline}")
print(f"After flood: {ptu_routes} PTU, {payg_routes} PAYG")

if baseline == 'ptu-pool' and payg_routes > 0:
    print("\n   ✓ P2 shifted from PTU to PAYG as utilization rose")
    print("P2 dynamic routing works! ✅")
elif payg_routes == len(p2_routes):
    print("\n   P2 went straight to PAYG — utilization was already high")
else:
    print("\n   P2 stayed on PTU — simulated utilization may not have crossed 70%")

---
## Test Summary

In [ ]:
summary = """
| # | Test | What it verifies |
|---|------|------------------|
| 1 | Config Viewer | `/gateway/config` renders HTML with contract details |
| 2 | Identity Resolution | Each team gets correct contract (tpm, quota, ptu limits) |
| 3 | Priority Routing | All priorities route to pool, counters decrease correctly |
| 4 | Model Access Control | Unlisted models get 403 |
| 5 | Unknown Caller | Invalid/unregistered tokens get 401 |
| 6 | Per-Model TPM | Rapid requests hit 429 at TPM limit |
| 7 | Monthly Quota | Cumulative tokens exhaust monthly cap |
| 8 | PTU Soft Overflow | PTU exhausted → requests overflow to PAYG (x-backend-type changes from ptu to payg) |
| 9 | P2 Under Load | P2 shifts to PAYG when PTU utilization > 70% |
| 10 | Cross-Team Isolation | One team's rate limit doesn't affect others |
| 11 | PTU Simulator 429 | Simulated PTU returns 429 when capacity exhausted |
| 12 | P2 Dynamic Shift | P2 moves PTU→PAYG as simulated utilization rises |
| 13 | Header Completeness | All custom gateway headers present on 200 and error responses |
"""
display(Markdown(summary))

---
## Test 13: Response Header Completeness

Verify that all expected custom gateway headers are present and non-empty on a successful 200 response.

In [ ]:
print("=== Response Header Completeness ===\n")

# Expected headers on a 200 response
EXPECTED_200_HEADERS = [
    'x-ratelimit-limit-tokens',
    'x-ratelimit-remaining-tokens',
    'x-quota-limit-tokens',
    'x-quota-remaining-tokens',
    'x-caller-name',
    'x-caller-identity',
    'x-caller-priority',
    'x-route-target',
    'x-backend-type',
    'x-ptu-utilization',
    'x-ptu-consumed',
    'x-ptu-limit',
    'x-actual-tokens',
    'x-tokens-consumed',
]

# Expected headers on a 401/403/on-error response
EXPECTED_ERROR_HEADERS = [
    'x-error-reason',
    'x-error-message',
    'x-error-section',
    'x-error-source',
]

# --- Test 200 headers ---
print("Step 1: Verify all headers present on 200 response (Alpha)\n")
token = get_token('alpha')
url = f"{API_URL}/deployments/{MODEL}/chat/completions?api-version={API_VERSION}"
r = requests.post(url, headers={'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'},
                  json={'messages': [{'role': 'user', 'content': 'test'}], 'max_tokens': 5}, timeout=30)
assert r.status_code == 200, f"Expected 200, got {r.status_code}"

missing = []
empty = []
for h in EXPECTED_200_HEADERS:
    val = r.headers.get(h)
    if val is None:
        missing.append(h)
    elif val.strip() == '':
        empty.append(h)
    else:
        print(f"   ✓ {h}: {val}")

assert not missing, f"Missing headers: {missing}"
assert not empty, f"Empty headers: {empty}"
print(f"\n   All {len(EXPECTED_200_HEADERS)} headers present and non-empty ✅")

# --- Test error headers ---
print("\nStep 2: Verify error headers on 401 response (invalid token)\n")
r2 = requests.post(url, headers={'Authorization': 'Bearer invalid-token', 'Content-Type': 'application/json'},
                   json={'messages': [{'role': 'user', 'content': 'test'}], 'max_tokens': 5}, timeout=10)
assert r2.status_code == 401, f"Expected 401, got {r2.status_code}"

missing_err = []
for h in EXPECTED_ERROR_HEADERS:
    val = r2.headers.get(h)
    if val is None:
        missing_err.append(h)
    else:
        print(f"   ✓ {h}: {val}")

assert not missing_err, f"Missing error headers: {missing_err}"
print(f"\n   All {len(EXPECTED_ERROR_HEADERS)} error headers present ✅")

print("\nHeader completeness tests passed! ✅")